# FinanceBench Dataset Evaluation

In [ ]:
import torch
import json
import time
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import ast
from datasets import load_dataset
from mlx_lm import load, generate
import pandas as pd
import numpy as np
from tqdm import tqdm

Libraries imported successfully


## Load the FinanceBench Dataset

In [2]:
print("Loading FinanceBench dataset...")
ds = load_dataset("PatronusAI/financebench")
print(f"Dataset loaded: {ds}")
print(f"\nDataset features: {ds['train'].features.keys()}")
print(f"\nTotal examples in train split: {len(ds['train'])}")

# Filter for Information Extraction entries only
print("\nFiltering for 'Information extraction' question_reasoning only...")
filtered_data = []
for example in ds['train']:
    if example.get('question_reasoning') == 'Information extraction':
        filtered_data.append(example)

print(f"Filtered examples: {len(filtered_data)} (from {len(ds['train'])} total)")

# Inspect a sample entry
if filtered_data:
    sample = filtered_data[0]
    print("\n--- Sample Entry ---")
    print(f"Company: {sample['company']}")
    print(f"Question: {sample['question']}")
    print(f"Answer: {sample['answer']}")
    print(f"Question Type: {sample['question_type']}")
    print(f"Question Reasoning: {sample['question_reasoning']}")

Loading FinanceBench dataset...
Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})

Dataset features: dict_keys(['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'])

Total examples in train split: 150

Filtering for 'Information extraction' question_reasoning only...
Filtered examples: 31 (from 150 total)

--- Sample Entry ---
Company: 3M
Question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Answer: $1577.00
Que

In [3]:
# Explore dataset metadata
companies = defaultdict(int)
question_types = defaultdict(int)
doc_types = defaultdict(int)

for example in filtered_data:
    companies[example.get('company', 'Unknown')] += 1
    question_types[example.get('question_type', 'Unknown')] += 1
    doc_types[example.get('doc_type', 'Unknown')] += 1

print("\n--- Filtered Dataset Statistics ---")
print(f"Unique companies: {len(companies)}")
print(f"Top 10 companies: {dict(sorted(companies.items(), key=lambda x: x[1], reverse=True)[:10])}")
print(f"\nQuestion Types: {dict(question_types)}")
print(f"Document Types: {dict(doc_types)}")


--- Filtered Dataset Statistics ---
Unique companies: 18
Top 10 companies: {'PepsiCo': 4, '3M': 3, 'Amcor': 2, 'AMD': 2, 'American Express': 2, 'Best Buy': 2, 'Boeing': 2, 'CVS Health': 2, 'MGM Resorts': 2, 'Ulta Beauty': 2}

Question Types: {'metrics-generated': 14, 'domain-relevant': 17}
Document Types: {'10k': 30, '10q': 1}


## Loading the model

In [4]:
# Device detection
if torch.cuda.is_available():
    device = "cuda"
    model_name = "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF"
    print("CUDA detected, using GPU model")
elif torch.backends.mps.is_available():
    device = "mps"
    model_name = "lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit"
    print("Apple Silicon MPS detected, using MLX-optimized model")
else:
    device = "cpu"
    model_name = "lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit"
    print("CPU only, using MLX model")

print(f"Device: {device}")
print(f"Model: {model_name}")

Apple Silicon MPS detected, using MLX-optimized model
Device: mps
Model: lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit


In [5]:
print("Loading model and tokenizer...")
model, tokenizer = load(model_name)
print("Model loaded successfully!")

Loading model and tokenizer...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model loaded successfully!


## Model-Based Validator

In [6]:
def validate_answer(model, tokenizer, question: str, expected_answer: str, predicted_answer: str) -> bool:
    """Use the model to validate if the predicted answer is correct."""
    validation_prompt = f"""You are an expert financial analyst validating answers to financial questions. Given a question, an expected answer, and a predicted answer, determine if the predicted answer is correct.

Be strict in validation:
- If the predicted answer says "I don't know" or "cannot be determined" BUT the expected answer is a specific value, mark as INCORRECT.
- If both are evasive/negative, they may be correct.
- If the predicted answer provides a specific value, check if it matches or is semantically equivalent to the expected answer.

Question: {question}
Expected Answer: {expected_answer}
Predicted Answer: {predicted_answer}

Is the predicted answer correct? Answer with only YES or NO."""
    
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": validation_prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
        )
    else:
        formatted_prompt = validation_prompt
    
    response = generate(
        model,
        tokenizer,
        prompt=formatted_prompt,
        max_tokens=16,
        verbose=False,
    )
    
    response = response.strip().upper()
    return "YES" in response

print("Model-based validator defined")

Model-based validator defined


## Generate Model Predictions

In [7]:
def mlx_generate_compat(model, tokenizer, prompt: str, max_tokens: int = 512) -> str:
    """
    Generate text using MLX model with chat template support.
    """
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
        )
    
    response = generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=False,
    )
    return response.strip()

# Configuration for evaluation
NUM_SAMPLES = 100  # Evaluate on first 100 samples; set to -1 for all
MAX_TOKENS = 512
PROMPT_TEMPLATE = """Answer the following financial question to the best of your ability.

Question: {question}

Answer:"""

print(f"Evaluation configuration:")
print(f"- Available samples: {len(filtered_data)}")
print(f"- Requested samples: {NUM_SAMPLES if NUM_SAMPLES > 0 else len(filtered_data)}")
print(f"- Max tokens per generation: {MAX_TOKENS}")

Evaluation configuration:
- Available samples: 31
- Requested samples: 100
- Max tokens per generation: 512


In [8]:
# Run evaluation
results = []
num_eval_samples = min(NUM_SAMPLES, len(filtered_data)) if NUM_SAMPLES > 0 else len(filtered_data)

print(f"\nGenerating predictions on {num_eval_samples} samples...")
print("=" * 80)

for idx in tqdm(range(num_eval_samples), desc="Evaluating"):
    example = filtered_data[idx]
    question = example['question']
    ground_truth = example['answer']
    
    # Generate prompt
    prompt = PROMPT_TEMPLATE.format(question=question)
    
    # Generate prediction
    start_time = time.time()
    prediction = mlx_generate_compat(model, tokenizer, prompt, MAX_TOKENS)
    generation_time = time.time() - start_time
    
    # Store results
    result = {
        'index': idx,
        'company': example.get('company', 'Unknown'),
        'question': question,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'question_type': example.get('question_type', 'Unknown'),
        'doc_type': example.get('doc_type', 'Unknown'),
        'generation_time': generation_time,
    }
    results.append(result)

print("=" * 80)
print(f"Predictions generated for {len(results)} samples")
print(f"Average generation time: {np.mean([r['generation_time'] for r in results]):.3f}s")


Generating predictions on 31 samples...


Evaluating: 100%|██████████| 31/31 [01:31<00:00,  2.95s/it]

Predictions generated for 31 samples
Average generation time: 2.950s


## Results

In [9]:
# Compute evaluation metrics for each result
print("Validating predictions using model-based validator...")

for result in tqdm(results, desc="Validating"):
    question = result['question']
    ground_truth = result['ground_truth']
    prediction = result['prediction']
    
    is_correct = validate_answer(model, tokenizer, question, ground_truth, prediction)
    result['is_correct'] = int(is_correct)

# Convert to DataFrame for easier analysis
df_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("OVERALL EVALUATION RESULTS")
print("=" * 80)
print(f"\nTotal samples evaluated: {len(df_results)}")
print(f"\nMetrics Summary:")
print(f"  Accuracy (Model-Validated): {df_results['is_correct'].mean():.4f} ({df_results['is_correct'].sum()}/{len(df_results)})")
print(f"  Avg Generation Time: {df_results['generation_time'].mean():.3f}s")

Validating predictions using model-based validator...


Validating: 100%|██████████| 31/31 [00:24<00:00,  1.25it/s]


OVERALL EVALUATION RESULTS

Total samples evaluated: 31

Metrics Summary:
  Accuracy (Model-Validated): 0.2903 (9/31)
  Avg Generation Time: 2.950s


In [12]:
# Save results to file
output_file = Path("financebench_evaluation_results.jsonl")
print(f"\nSaving results to {output_file}...")

with open(output_file, 'w') as f:
    for result in results:
        f.write(json.dumps(result) + '\n')

print(f"Results saved to {output_file}")


Saving results to financebench_evaluation_results.jsonl...
Results saved to financebench_evaluation_results.jsonl
